# IT3212 – Assignment 3 – Task 7: Bagging and Boosting

This notebook builds on **Task 4 (Feature Selection)** and **Tasks 5–6 (Classification and Comparison)**. Task 7 implements two ensemble learning methods using the same 18 selected input features and the existing training/validation split.

**Models:** Random Forest (Bagging) and AdaBoost (Boosting).

**Target:** `smoking` (`0` = never smoked, `1` = current or former smoker).

**Inputs:** `../data/assignment3/task4_train.csv`, `task4_val.csv`, and `task4_metadata.json`.

Both models are tuned using the validation set, with ROC-AUC as the selection metric. The test set is reserved for the final evaluation after model decisions have been made. Performance is evaluated using accuracy, F1-score, ROC-AUC, precision and recall, with a classification threshold of 0.5.

In [1]:
# Imports and configuration
import json
import time
from pathlib import Path

import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 18
THRESHOLD = 0.5
DATA_DIR = Path("../data/assignment3")

In [2]:
# Load the features selected in Task 4 and the existing data splits
with open(DATA_DIR / "task4_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

FEATURES = metadata["selected_features"]
TARGET = metadata["target"]

train_df = pd.read_csv(DATA_DIR / "task4_train.csv")
val_df = pd.read_csv(DATA_DIR / "task4_val.csv")
test_df = pd.read_csv(DATA_DIR / "task4_test.csv")

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"Number of features: {len(FEATURES)}")
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")

Number of features: 18
Training set:   (787625, 18)
Validation set: (98453, 18)
Test set:       (98454, 18)


### Bagging: Random Forest

In [3]:
# Tune the number of trees using validation AUC
rf_candidates = []

rf_tuning_start = time.perf_counter()

for n_estimators in [50, 100]:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        min_samples_leaf=50,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_prob)

    rf_candidates.append({
        "model": model,
        "n_estimators": n_estimators,
        "validation_auc": val_auc,
    })

random_forest_tuning_time = time.perf_counter() - rf_tuning_start

best_rf = max(rf_candidates, key=lambda result: result["validation_auc"])
random_forest_model = best_rf["model"]

print("Random Forest – tuning results")
for result in rf_candidates:
    print(
        f'{result["n_estimators"]} trees: '
        f'validation AUC = {result["validation_auc"]:.4f}'
    )

print("\nSelected configuration:")
print(f'Trees: {best_rf["n_estimators"]}')
print(f'Validation AUC: {best_rf["validation_auc"]:.4f}')
print(f'Tuning and training time: {random_forest_tuning_time:.2f} seconds')

Random Forest – tuning results
50 trees: validation AUC = 0.8675
100 trees: validation AUC = 0.8676

Selected configuration:
Trees: 100
Validation AUC: 0.8676
Tuning and training time: 26.98 seconds


### Boosting: AdaBoost

In [4]:
# Tune the number of estimators and learning rate using validation AUC
ada_candidates = []

ada_tuning_start = time.perf_counter()

for n_estimators in [25, 50]:
    for learning_rate in [0.5, 1.0]:
        model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=1,
                random_state=RANDOM_STATE,
            ),
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=RANDOM_STATE,
        )

        model.fit(X_train, y_train)
        val_prob = model.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)

        ada_candidates.append({
            "model": model,
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "validation_auc": val_auc,
        })

ada_tuning_time = time.perf_counter() - ada_tuning_start

best_ada = max(ada_candidates, key=lambda result: result["validation_auc"])
boosting_model = best_ada["model"]

print("AdaBoost – tuning results")
for result in ada_candidates:
    print(
        f'{result["n_estimators"]} estimators, '
        f'learning rate {result["learning_rate"]}: '
        f'validation AUC = {result["validation_auc"]:.4f}'
    )

print("\nSelected configuration:")
print(f'Estimators: {best_ada["n_estimators"]}')
print(f'Learning rate: {best_ada["learning_rate"]}')
print(f'Validation AUC: {best_ada["validation_auc"]:.4f}')
print(f'Tuning and training time: {ada_tuning_time:.2f} seconds')

AdaBoost – tuning results
25 estimators, learning rate 0.5: validation AUC = 0.8645
25 estimators, learning rate 1.0: validation AUC = 0.8699
50 estimators, learning rate 0.5: validation AUC = 0.8699
50 estimators, learning rate 1.0: validation AUC = 0.8729

Selected configuration:
Estimators: 50
Learning rate: 1.0
Validation AUC: 0.8729
Tuning and training time: 103.55 seconds


### Final evaluation on the test set

In [5]:
# Calculate test metrics for a fitted model
def evaluate_model(model, X, y):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= THRESHOLD).astype(int)

    return {
        "Accuracy": accuracy_score(y, predictions),
        "Precision": precision_score(y, predictions),
        "Recall": recall_score(y, predictions),
        "F1-score": f1_score(y, predictions),
        "ROC-AUC": roc_auc_score(y, probabilities),
    }


rf_test_metrics = evaluate_model(random_forest_model, X_test, y_test)
ada_test_metrics = evaluate_model(boosting_model, X_test, y_test)

print("Test set results (threshold = 0.5)")
print("----------------------------------")
print("Random Forest:", {k: round(v, 4) for k, v in rf_test_metrics.items()})
print("AdaBoost:     ", {k: round(v, 4) for k, v in ada_test_metrics.items()})

Test set results (threshold = 0.5)
----------------------------------
Random Forest: {'Accuracy': 0.8092, 'Precision': 0.6958, 'Recall': 0.9118, 'F1-score': 0.7893, 'ROC-AUC': 0.87}
AdaBoost:      {'Accuracy': 0.8092, 'Precision': 0.7049, 'Recall': 0.8831, 'F1-score': 0.784, 'ROC-AUC': 0.8748}


### Model size and training time

In [6]:
# Measure the fitted ensemble sizes
rf_depths = [tree.get_depth() for tree in random_forest_model.estimators_]
ada_depths = [tree.get_depth() for tree in boosting_model.estimators_]

results = pd.DataFrame([
    {
        "Model": "Random Forest",
        **rf_test_metrics,
        "Training time (s)": random_forest_tuning_time,
        "Number of trees": len(random_forest_model.estimators_),
        "Maximum tree depth": max(rf_depths),
    },
    {
        "Model": "AdaBoost",
        **ada_test_metrics,
        "Training time (s)": ada_tuning_time,
        "Number of trees": len(boosting_model.estimators_),
        "Maximum tree depth": max(ada_depths),
    },
])

results.round(4)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,Training time (s),Number of trees,Maximum tree depth
0,Random Forest,0.8092,0.6958,0.9118,0.7893,0.8700,26.9750,100,5
1,AdaBoost,0.8092,0.7049,0.8831,0.7840,0.8748,103.5532,50,1


### AdaBoost estimator inspection

In [7]:
print("First 10 AdaBoost estimator errors:")
print(boosting_model.estimator_errors_[:10].round(4))

print("\nFirst 10 AdaBoost estimator weights:")
print(boosting_model.estimator_weights_[:10].round(4))

First 10 AdaBoost estimator errors:
[0.1937 0.297  0.4041 0.4452 0.467  0.4688 0.4771 0.4768 0.468  0.4722]

First 10 AdaBoost estimator weights:
[1.4259 0.8616 0.3884 0.2202 0.1321 0.1249 0.0917 0.0929 0.1282 0.1112]
